<a href="https://colab.research.google.com/github/candeperles/PDI-TP/blob/main/notebooks/05_clasificacion_resnet50.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ResNet50 · Colab v2

Clasificación de paneles solares: ImageNet congelado, fine-tuning de conv5 y pesos aleatorios.

**Empezá en una sesión de Colab con GPU T4 y ejecutá de arriba hacia abajo.**

1. Autorizá el acceso a Drive cuando Colab lo pida (`drive.mount`, igual que el resto de los notebooks).

Cada variante se guarda con el formato común del equipo en `Mi unidad/PDI-TP/resultados/<modelo>/<modo>/`:
`resnet50/imagenet_cabeza`, `resnet50/imagenet_finetuning`, `resnet50/aleatorio` y `baseline/siempre_con_paneles`.
Dentro de cada una: pesos (`mejor.keras`, y `ultimo.keras` para reanudar), `metricas.json`,
`predicciones_val.csv` (predicción contra el real), `config.json`, historial, curvas y matriz de confusión.

Los checkpoints se escriben directo en Drive, así que si se corta la sesión se puede volver a ejecutar todo:
las variantes completas se saltean y una etapa a medias continúa desde `ultimo.keras`. `REPETIR = True`
vuelve a entrenar (la corrida anterior se archiva como `<modo>__anterior_<fecha>`, no se borra).

El split y los datos originales se conservan. Las referencias N06/N16 siguen siendo provisionales; se informa
además la evaluación sin esos casos. Val selecciona checkpoints: no es un test independiente.


In [ ]:
# 1. Opciones: solo hace falta cambiar estos valores.
REPETIR = False  # True vuelve a entrenar variantes que ya están completas (archiva las anteriores)

SEMILLA = 42
LADO = 256
BATCH = 16
EPOCAS_CABEZA = 10
EPOCAS_FINE = 10
EPOCAS_ALEATORIO = 20
UMBRAL = 0.5
AUGMENTATION = True


In [ ]:
# 2. Preparar dependencias, código compartido y Drive.
import importlib.util
import subprocess
import sys
import os
from pathlib import Path

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Este notebook está preparado para Google Colab. Abrilo allí con GPU T4.")
for paquete, modulo in [
    ("tensorflow>=2.20,<2.22", "tensorflow"), ("numpy", "numpy"),
    ("pandas", "pandas"), ("Pillow", "PIL"), ("matplotlib", "matplotlib"),
    ("scikit-learn", "sklearn")]:
    if importlib.util.find_spec(modulo) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

repo = Path("/content/PDI-TP")
if repo.is_dir():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
os.chdir(repo)
sys.path.insert(0, str(repo))

import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from importlib.metadata import version
from pdi_tp import entorno, resultados, clasificacion as clf

rutas = entorno.preparar()  # monta Drive
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("Activá GPU T4 en Entorno de ejecución > Cambiar tipo de entorno y volvé a empezar.")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
keras.utils.set_random_seed(SEMILLA)
tf.config.experimental.enable_op_determinism()
print("Python:", platform.python_version(), "TensorFlow:", tf.__version__, "GPU:", gpus)

MODELO = "resnet50"  # carpetas en Drive: resultados/resnet50/<modo>/
ANALISIS = rutas.analisis / "resnet50"  # manifiesto y cajas derivadas (no son resultados de un modelo)
ANALISIS.mkdir(parents=True, exist_ok=True)


## Referencia y preparación reproducible

Se auditan los 900 pares. Solo se permite excluir Z01/línea 1, de ancho cero, según criterios v1.
Se guardan un manifiesto binario y metadatos por caja, con clase y confianza originales.
No se necesita copiar imágenes para clasificación. Las cajas positivas pequeñas y las clases 1/2 se conservan.

**N06 y N16 siguen pendientes**: `y` permanece vacío y `y_original=0` conserva lo entregado.
Se guardan predicciones para las 162 imágenes. Las métricas completas y la selección de checkpoints
usan provisionalmente la referencia original; se añade análisis de sensibilidad sobre los 160 casos
sin duda registrada. Esto no confirma semánticamente las otras etiquetas ni reemplaza la segunda revisión.

In [ ]:
tabla, cajas = clf.preparar_tabla(rutas)
tabla.to_csv(ANALISIS / "manifiesto.csv", index=False)
cajas.to_csv(ANALISIS / "cajas_derivadas.csv", index=False)
train = tabla.query("split == 'train'").reset_index(drop=True)
val = tabla.query("split == 'val'").reset_index(drop=True)
PESOS = clf.pesos_clase(train.y.to_numpy(dtype=int))  # calculados únicamente con train
HUELLA = clf.huella_tabla(tabla)
display(tabla.groupby(["split", "y_original", "pendiente_revision"]).size().rename("imagenes").to_frame())
display(val.loc[val.pendiente_revision, ["archivo", "y", "y_original"]])
print("Pesos de clase:", PESOS, "\nHuella:", HUELLA)
config = dict(modelo="ResNet50", lado=LADO, batch=BATCH, semilla=SEMILLA,
    augmentation=AUGMENTATION, aumentos="flip horizontal/vertical + rotacion 0.25 vueltas",
    dropout=0.3, cabeza="GlobalAveragePooling2D, Dropout, Dense sigmoid",
    umbral=UMBRAL, clase_positiva=1, promedio="binary", pesos_clase=PESOS,
    criterio_datos="v1; excluir solo Z01 linea 1; conservar cajas pequenas y confianza",
    huella_manifiesto=HUELLA, n_train=len(train), n_val=len(val),
    evaluacion="provisional; referencia original; N06/N16 pendientes",
    monitor="val_loss", patience=5, optimizador="Adam", loss="binary_crossentropy",
    preprocesamiento="ResNet preprocess_input sobre RGB 0..255, incluido en modelo",
    epocas_cabeza=EPOCAS_CABEZA, epocas_fine=EPOCAS_FINE, epocas_aleatorio=EPOCAS_ALEATORIO,
    entorno=dict(python=platform.python_version(),
                 paquetes={p: version(p) for p in ["tensorflow", "keras", "numpy", "pandas", "Pillow", "scikit-learn"]},
                 gpu=[str(g) for g in gpus]))


## Modelo y checkpoints

Keras escribe los checkpoints directo en la carpeta de Drive de cada variante. Volver a ejecutar una etapa
finalizada carga su mejor checkpoint; una etapa incompleta continúa desde `ultimo.keras`, conservando Adam,
las épocas del historial y el mejor checkpoint, aunque sea en otra sesión de Colab.


In [ ]:
"""Entrenamiento ResNet50 con TensorFlow/Keras; importar en el entorno de modelos."""
import numpy as np
import tensorflow as tf
from tensorflow import keras


def dataset(tabla, lado=256, batch=32, entrenamiento=False, semilla=42):
    """RGB 0..255 -> preprocess_input de ResNet; val conserva el orden del manifiesto."""
    ds = tf.data.Dataset.from_tensor_slices((tabla.ruta.to_list(), tabla.y_original.to_numpy(dtype="float32")))
    if entrenamiento:
        ds = ds.shuffle(len(tabla), seed=semilla, reshuffle_each_iteration=True)

    def leer(ruta, etiqueta):
        imagen = tf.io.decode_png(tf.io.read_file(ruta), channels=3)
        imagen = tf.image.resize(imagen, (lado, lado))
        return imagen, tf.reshape(etiqueta, (1,))

    return ds.map(leer, num_parallel_calls=tf.data.AUTOTUNE).batch(batch).prefetch(tf.data.AUTOTUNE)


def construir(lado=256, preentrenado=True, augmentation=True, semilla=42):
    base = keras.applications.ResNet50(include_top=False,
        weights="imagenet" if preentrenado else None, input_shape=(lado, lado, 3))
    base.trainable = not preentrenado
    entrada = keras.Input((lado, lado, 3), name="rgb_0_255")
    x = entrada
    if augmentation:
        x = keras.Sequential([
            keras.layers.RandomFlip("horizontal_and_vertical", seed=semilla),
            keras.layers.RandomRotation(0.25, seed=semilla + 1),
        ], name="augmentation")(x)
    # ResNet espera BGR centrado en medias ImageNet. No dividir por 255.
    x = keras.applications.resnet.preprocess_input(x)
    # ImageNet: BatchNorm en inferencia incluso durante fine-tuning.
    # Desde cero: BatchNorm debe aprender sus estadísticas.
    x = base(x, training=False) if preentrenado else base(x)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3, seed=semilla + 2)(x)
    salida = keras.layers.Dense(1, activation="sigmoid", name="con_paneles")(x)
    return keras.Model(entrada, salida, name="clasificador_resnet50"), base


def descongelar(base):
    """Ajusta el último bloque residual conv5; mantiene BatchNormalization congelada."""
    base.trainable = True
    for capa in base.layers:
        capa.trainable = capa.name.startswith("conv5_") and not isinstance(capa, keras.layers.BatchNormalization)


def compilar(modelo, lr):
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                   loss="binary_crossentropy", metrics=[keras.metrics.BinaryAccuracy(name="accuracy")])


class ParadaReanudada(keras.callbacks.EarlyStopping):
    def __init__(self, valores):
        super().__init__(monitor="val_loss", mode="min", patience=5, restore_best_weights=False)
        self.valores_previos = list(valores)

    def on_train_begin(self, logs=None):
        super().on_train_begin(logs)
        if self.valores_previos:
            self.best = min(self.valores_previos)
            self.wait = len(self.valores_previos) - 1 - int(np.argmin(self.valores_previos))


def entrenar(modelo, train_ds, val_ds, carpeta, epocas, lr, pesos):
    carpeta.mkdir(parents=True, exist_ok=True)
    terminado = carpeta / "completado.json"
    if terminado.exists():
        modelo.load_weights(carpeta / "mejor.keras")
        print("Etapa ya terminada; se cargó el mejor checkpoint:", carpeta.name)
        return modelo
    historial = carpeta / "historia.csv"
    ultimo = carpeta / "ultimo.keras"
    mejor = carpeta / "mejor.keras"
    previo = pd.read_csv(historial) if historial.exists() else pd.DataFrame()
    inicial = len(previo)
    valores = []
    if inicial:
        if not ultimo.exists() or not mejor.exists():
            raise RuntimeError("Hay historial pero falta un checkpoint. No se sobrescribió nada; conservá la sesión para recuperar los archivos.")
        if not np.array_equal(previo.epoch.to_numpy(), np.arange(inicial)) or not np.isfinite(previo.val_loss).all():
            raise RuntimeError("El historial está incompleto o no es finito. No se modificaron los checkpoints.")
        # El modelo completo conserva pesos y estado de Adam; no recompilarlo.
        modelo = keras.models.load_model(ultimo)
        if int(modelo.optimizer.iterations.numpy()) != inicial * len(train_ds):
            raise RuntimeError("El último checkpoint y el historial no coinciden. No se sobrescribió nada.")
        valores = previo.val_loss.to_list()
        print(f"Reanudando {carpeta.name} desde la época {inicial + 1}; se conservan las {inicial} anteriores.")
    else:
        if ultimo.exists() or mejor.exists():
            raise RuntimeError("Hay checkpoints sin historial de épocas; no se sobrescribieron.")
        compilar(modelo, lr)
    espera = len(valores) - 1 - int(np.argmin(valores)) if valores else 0
    if inicial < epocas and espera < 5:
        historia = modelo.fit(train_ds, validation_data=val_ds, epochs=epocas,
            initial_epoch=inicial, class_weight=pesos, callbacks=[
                keras.callbacks.TerminateOnNaN(),
                keras.callbacks.ModelCheckpoint(str(mejor), monitor="val_loss", mode="min",
                    save_best_only=True, initial_value_threshold=min(valores) if valores else None),
                keras.callbacks.ModelCheckpoint(str(ultimo)),
                keras.callbacks.CSVLogger(str(historial), append=bool(inicial)),
                ParadaReanudada(valores),
            ], verbose=2)
        if not all(np.isfinite(v).all() for v in historia.history.values()):
            raise RuntimeError("Entrenamiento no finito. No se registra como completado.")
    modelo.load_weights(mejor)
    terminado.write_text(json.dumps({"epocas": len(pd.read_csv(historial)), "monitor": "val_loss"}))
    return modelo


from types import SimpleNamespace
rn = SimpleNamespace(dataset=dataset, construir=construir, descongelar=descongelar, entrenar=entrenar)


## Evaluación común y baseline

La matriz usa filas = referencia, columnas = predicción, orden `[sin paneles, con paneles]`.
Se exporta una fila por imagen con nombre, fuente, grupo, referencia, probabilidad y predicción
para la futura comparación con los detectores. Se informa también el baseline «siempre con paneles».
Los archivos `.keras` incluyen preprocesamiento y esperan imágenes RGB en 0..255.

In [ ]:
def evaluar_guardar(carpeta, probabilidades, params, modelo=None):
    """Guarda la variante con el formato común (resultados.guardar_corrida) y agrega sus extras."""
    p = np.asarray(probabilidades).reshape(-1)
    mascara = ~val.pendiente_revision.to_numpy()
    extra = {"sin_pendientes": clf.evaluar(val.loc[mascara, "y_original"].to_numpy(dtype=int), p[mascara], UMBRAL)}
    if modelo is not None:
        extra["parametros_modelo"] = dict(total=modelo.count_params(),
            entrenables=int(sum(np.prod(w.shape) for w in modelo.trainable_weights)))
    metricas = resultados.guardar_corrida(rutas, carpeta, archivos=val.archivo, probabilidad=p,
        params=dict(config, **params), umbral=UMBRAL, metricas_extra=extra, sin_pesos=modelo is None)
    from sklearn.metrics import ConfusionMatrixDisplay
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(np.array(metricas["matriz_confusion"]),
        display_labels=["sin paneles", "con paneles"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"{carpeta.parent.name}/{carpeta.name} · referencia provisional")
    fig.tight_layout()
    fig.savefig(carpeta / "matriz_confusion.png", dpi=130)
    plt.show()
    plt.close(fig)
    return pd.read_csv(carpeta / "predicciones_val.csv")

def curvas(carpeta):
    historia = pd.read_csv(carpeta / "historia.csv")
    fig, axs = plt.subplots(1, 2, figsize=(10, 3))
    for ax, metrica in zip(axs, ["loss", "accuracy"]):
        ax.plot(historia.epoch + 1, historia[metrica], label="train")
        ax.plot(historia.epoch + 1, historia["val_" + metrica], label="val")
        ax.set(xlabel="Época", ylabel=metrica, title=carpeta.name)
        ax.legend()
    fig.tight_layout()
    fig.savefig(carpeta / "curvas.png", dpi=130)
    plt.show()
    plt.close(fig)

def preparar_variante(modelo, modo):
    """Carpeta de la variante y si ya está completa (en ese caso se reutiliza, salvo REPETIR)."""
    carpeta = resultados.ruta_corrida(rutas, modelo, modo)
    if resultados.corrida_completa(carpeta) and not REPETIR:
        print(f"Ya completa, se reutiliza: {modelo}/{modo}")
        return carpeta, True
    return resultados.carpeta_corrida(rutas, modelo, modo, reiniciar=REPETIR), False

carpeta, hecha = preparar_variante("baseline", "siempre_con_paneles")
if not hecha:
    evaluar_guardar(carpeta, np.ones(len(val)), dict(estrategia="baseline"))


## 1. ImageNet: entrenar solo la cabeza

El extractor queda congelado. `preprocess_input` convierte RGB a BGR y resta las medias de ImageNet;
no dividir los píxeles por 255. Los aumentos solo se activan al entrenar.

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEMILLA)
train_ds = rn.dataset(train, LADO, BATCH, entrenamiento=True, semilla=SEMILLA)
val_ds = rn.dataset(val, LADO, BATCH)
modelo, base = rn.construir(LADO, preentrenado=True, augmentation=AUGMENTATION, semilla=SEMILLA)
modelo.summary()
carpeta, hecha = preparar_variante(MODELO, "imagenet_cabeza")
if hecha:
    modelo.load_weights(carpeta / "mejor.keras")  # la etapa 2 parte de este checkpoint
else:
    modelo = rn.entrenar(modelo, train_ds, val_ds, carpeta, EPOCAS_CABEZA, 1e-3, PESOS)
    evaluar_guardar(carpeta, modelo.predict(val_ds, verbose=0),
        dict(estrategia="cabeza", inicializacion="imagenet", lr=1e-3, epocas_max=EPOCAS_CABEZA), modelo)
    curvas(carpeta)
CARPETA_CABEZA = carpeta


## 2. ImageNet: ajuste fino del bloque conv5

Continúa desde el mejor checkpoint de la cabeza, que ya fue restaurado. Se descongela conv5
excepto BatchNorm, se recompila con Adam nuevo y LR 1e-5. La base se llama con `training=False`.
Se registra la dependencia del experimento anterior y las épocas adicionales.

In [ ]:
base = modelo.get_layer("resnet50")
rn.descongelar(base)
carpeta, hecha = preparar_variante(MODELO, "imagenet_finetuning")
if not hecha:
    modelo = rn.entrenar(modelo, train_ds, val_ds, carpeta, EPOCAS_FINE, 1e-5, PESOS)
    evaluar_guardar(carpeta, modelo.predict(val_ds, verbose=0),
        dict(estrategia="conv5_sin_batchnorm", inicializacion="mejor_imagenet_cabeza",
             checkpoint_inicial=str(CARPETA_CABEZA / "mejor.keras"),
             lr=1e-5, epocas_adicionales_max=EPOCAS_FINE), modelo)
    curvas(carpeta)
del modelo, base, train_ds, val_ds


## 3. ResNet50 desde cero

Se crea una red nueva con `weights=None`, sin reutilizar pesos anteriores. Toda la red se entrena;
BatchNorm actualiza sus estadísticas. Se mantienen cabeza, aumentos y preprocesamiento para comparar.
Con 738 imágenes puede sobreajustar y necesitar mayor presupuesto; interpretar también las curvas.

In [ ]:
carpeta, hecha = preparar_variante(MODELO, "aleatorio")
if not hecha:
    keras.backend.clear_session()
    keras.utils.set_random_seed(SEMILLA)
    train_ds = rn.dataset(train, LADO, BATCH, entrenamiento=True, semilla=SEMILLA)
    val_ds = rn.dataset(val, LADO, BATCH)
    modelo, base = rn.construir(LADO, preentrenado=False, augmentation=AUGMENTATION, semilla=SEMILLA)
    modelo = rn.entrenar(modelo, train_ds, val_ds, carpeta, EPOCAS_ALEATORIO, 1e-4, PESOS)
    evaluar_guardar(carpeta, modelo.predict(val_ds, verbose=0),
        dict(estrategia="toda_la_red", inicializacion="aleatoria", lr=1e-4, epocas_max=EPOCAS_ALEATORIO), modelo)
    curvas(carpeta)


## Comparar e interpretar los resultados reales

La tabla se completa únicamente al ejecutar los entrenamientos. No hay métricas simuladas.
Comparar F1 por clase y F1 macro con el baseline, no solo accuracy. Revisar si los errores
se concentran por fuente, en cajas pequeñas o en anotaciones inciertas. La mejora con ajuste fino
debe acompañarse de curvas y presupuesto total; una corrida con semilla fija no mide variabilidad.
Falta repetir con otra arquitectura para completar el requisito global del ítem 2.

In [ ]:
corridas = resultados.cargar_corridas(rutas)
display(corridas[corridas.modelo.isin([MODELO, "baseline"])]
        .set_index(["modelo", "modo"])[["accuracy", "precision", "recall", "f1", "f1_macro", "recall_sin_paneles"]])
# Todas las predicciones mantienen los nombres para comparar con YOLO sobre el mismo val.
predicciones = pd.read_csv(resultados.ruta_corrida(rutas, MODELO, "aleatorio") / "predicciones_val.csv")
display(predicciones.query("acierto == 0").head(20))
print("Resultados:", resultados.ruta_corrida(rutas, MODELO, "x").parent)
